# Qwen 3 Mathematical Reasoning Fine-Tuning with GRPO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/To-Data-Beyond/Generative-AI-Techanical-Tutorials/blob/main/Qwen_3_Mathematical_Reasoning_Fine_Tuning_with_GRPO.ipynb)

This notebook contains the complete code workflow for both parts of the To Data & Beyond tutorial:

- [Part 1: environment, model, LoRA, and dataset preparation](https://todatabeyond.com/blog/qwen-3-mathematical-reasoning-fine-tuning-with-grpo-technique-1)
- [Part 2: reward functions, GRPO training, testing, and model export](https://todatabeyond.com/blog/qwen-3-mathematical-reasoning-fine-tuning-with-grpo-technique-2)
- [Tutorial code repository](https://github.com/To-Data-Beyond/Generative-AI-Techanical-Tutorials)
- [View this notebook on GitHub](https://github.com/To-Data-Beyond/Generative-AI-Techanical-Tutorials/blob/main/Qwen_3_Mathematical_Reasoning_Fine_Tuning_with_GRPO.ipynb)


## Before you run the notebook

- Use a CUDA-capable GPU. A high-memory Colab runtime is recommended for Qwen3-4B GRPO training.
- Run the cells in order because Part 2 continues from the model and tokenizer prepared in Part 1.
- The example training run uses 100 GRPO steps. Increase the sequence length and training duration for stronger results.
- Keep Hugging Face tokens in Colab secrets or environment variables. Never commit access tokens to the notebook or repository.
- Package APIs can change over time; if a pinned version becomes incompatible, check the current Unsloth and TRL documentation.


## Part 1 — Prepare Qwen 3 for GRPO


Enhancing the reasoning abilities of Large Language Models (LLMs) is important for their application in complex tasks. This technical guide begins a practical walkthrough for converting the Qwen3 4B-Base model into a reasoning model through Group Relative Policy Optimization (GRPO) using OpenMathReasoning data.

As the first part in a two-part tutorial, this article focuses on the foundational steps required before starting the fine-tuning loop. It introduces GRPO, sets up the computational environment, loads the Qwen 3 base model and tokenizer, and acquires and prepares the target dataset. Completing these stages prepares the model for the reward functions and fine-tuning process covered in Part 2.

## Introduction to GRPO

GRPO is an advanced technique designed to improve the efficiency of fine-tuning large language models.

It combines reinforcement-learning principles with pretraining to refine a model’s behavior using reward signals rather than direct supervision. GRPO optimizes the model’s parameters iteratively through a policy-based optimization approach.

In a typical fine-tuning scenario, a model is trained on a supervised dataset and learns directly from ground-truth labels. GRPO instead introduces a reinforcement-learning paradigm in which the model is trained to maximize a reward signal that guides its behavior.

This process allows the model to adapt more flexibly to task-specific nuances, improving both accuracy and generalization.

The key formula for policy optimization in GRPO can be expressed as:

The terms in the policy objective are defined as follows:

This policy-based approach ensures that the model continuously adapts to the feedback provided during training, focusing on improving the reward signal that corresponds to task-specific goals.

In GRPO, the reward function can be defined according to specific task requirements, guiding the model toward the desired behavior. The reward can combine multiple factors such as accuracy, formatting, or logical consistency. For example, a correctness reward can be defined as:

This feedback mechanism allows GRPO to progressively refine the model, emphasizing the areas that matter most for the task.

## Setting Up the Working Environment

Before fine-tuning Qwen 3 with GRPO, we need to configure the environment with the required libraries. The following notebook cell installs the essential packages:


In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft "trl==0.15.2" triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt


Here is what the setup does:

1. %%capture is an IPython magic command used in Jupyter and Google Colab. It suppresses the cell output so lengthy installation logs do not clutter the notebook.
2. The base installation adds Unsloth and vLLM. The --no-deps flag prevents pip from automatically installing dependency versions that could conflict with packages already present in Colab.
3. The COLAB_ environment check detects Google Colab. The module-cleanup step removes potentially conflicting PIL and Google modules from Python’s cache before installing the intended versions.
4. The additional dependencies provide quantization, distributed execution, optimized attention, parameter-efficient fine-tuning, reinforcement-learning trainers, GPU kernels, tokenization, dataset loading, and Hugging Face Hub transfers.
5. The final step downloads vLLM’s requirements, removes transformers, numpy, and xformers from that list, and installs the remaining dependencies without overwriting the versions selected for this notebook.

### Core Libraries

- Unsloth speeds up LLM fine-tuning and lowers memory use, often making larger-model training possible on consumer GPUs.
- vLLM is a high-throughput engine for LLM inference and serving. Although this tutorial focuses on fine-tuning, it is also useful for efficient evaluation and later deployment.

### Additional Colab Dependencies

- bitsandbytes enables techniques such as 4-bit quantization and QLoRA, substantially reducing the model’s memory footprint.
- accelerate simplifies running PyTorch training across CPUs, GPUs, TPUs, and mixed-precision configurations.
- xformers supplies memory-efficient attention and other optimized Transformer building blocks. Version 0.0.29.post3 is pinned for compatibility.
- peft provides parameter-efficient methods such as LoRA, allowing a large model to be fine-tuned by updating only a small fraction of its parameters.
- trl provides reinforcement-learning and preference-tuning trainers, including the tools needed for GRPO. This environment pins version 0.15.2.
- triton, cut_cross_entropy, and unsloth_zoo provide lower-level kernels and optimization utilities used by the training stack.
- sentencepiece, protobuf, datasets, huggingface_hub, and hf_transfer handle tokenization, data loading, model and dataset access, and accelerated Hub transfers.

## Loading the Model and Tokenizer

With the environment ready, the next step is to load the Qwen 3 model. We use Unsloth’s FastLanguageModel class, which is engineered for faster loading and lower memory use than standard Hugging Face methods.

The following code loads the model and its tokenizer:


In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Base",
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.7, # Reduce if out of memory
)


We import FastLanguageModel from Unsloth and set max_seq_length to 2,048 tokens. FastLanguageModel.from_pretrained then loads the unsloth/Qwen3-4B-Base model.

Instead of training all of the model’s parameters—which would be computationally expensive and memory intensive—we use Low-Rank Adaptation (LoRA). LoRA injects small trainable adapter layers into the model, allowing strong fine-tuning results while updating only a small fraction of the total parameters.

Unsloth simplifies this process with FastLanguageModel.get_peft_model:


In [ ]:
lora_rank = 16
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)


After this code executes, the model becomes a PEFT model. Its original weights remain frozen while new trainable LoRA matrices are injected into the selected attention and MLP modules. During fine-tuning, only these adapters and any other explicitly unfrozen parameters are updated.

Unsloth optimizes this process for speed and memory efficiency. Parameters such as lora_alpha = lora_rank * 2 accelerate training, while use_gradient_checkpointing = "unsloth" reduces memory use. Together, these choices make fine-tuning powerful LLMs more accessible on commodity hardware.

## Loading and Preprocessing the Dataset

Because we are using a base model, we need to define a chat template. You can create your own template; here we follow DeepSeek’s idea of separating reasoning and the final answer, but use custom <start_working_out> and <end_working_out> tags.

We begin with a system prompt that guides the model toward a structured response for reasoning-intensive tasks:


In [ ]:
reasoning_start = "<start_working_out>" # Acts as <think>
reasoning_end   = "<end_working_out>"   # Acts as </think>
solution_start  = "<SOLUTION>"
solution_end    = "</SOLUTION>"

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""
system_prompt


Next, we define the chat template that tells the tokenizer how to convert a list of conversation messages into the single formatted string the language model expects:


In [ ]:
chat_template = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"\
    "{% endif %}"


The template applies the following logic:

1. Start with a system message. If the input includes one, use it; otherwise, use the default system prompt. Append the end-of-sequence token.
2. Iterate through the remaining messages. Append user content directly, and append assistant content followed by the end-of-sequence token.
3. If add_generation_prompt is true, append the reasoning-start marker so the model knows where to begin generating.

Instead of passing system_prompt and reasoning_start as variables every time apply_chat_template is called, we replace their placeholders with their current string values. This makes those parts of the Jinja template fixed by the Python setup:


In [ ]:
# Replace with our specific template:
chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{system_prompt}'")\
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")


Finally, we assign the modified template to the tokenizer. Later calls to tokenizer.apply_chat_template use this Jinja string to format message lists for the model:


In [ ]:
tokenizer.chat_template = chat_template


### Testing the Chat Template

For example, assume the end-of-sequence token is </s>, the system prompt is “You are a pirate,” reasoning_start is [THINK], and the user message is “Hello matey!” If add_generation_prompt is true, the simplified result would be:

The model would continue from [THINK]. An assistant reply would be terminated by </s> before the next message. This template focuses on message content and end-of-sequence boundaries rather than adding explicit “User:” or “Assistant:” prefixes.

We can now test the actual template with a short conversation:


In [ ]:
tokenizer.apply_chat_template([
    {"role" : "user", "content" : "What is 1+1?"},
    {"role" : "assistant", "content" : f"{reasoning_start}I think it's 2.{reasoning_end}{solution_start}2{solution_end}"},
    {"role" : "user", "content" : "What is 2+2?"},
], tokenize = False, add_generation_prompt = True)


### Loading OpenMathReasoning Data

We use a subset of NVIDIA’s OpenMathReasoning dataset that was filtered to include high-quality DeepSeek R1 traces. We retain approximately 59 examples to prime, or pre-fine-tune, the model so it learns the custom GRPO response format.

[View NVIDIA OpenMathReasoning on Hugging Face](https://huggingface.co/datasets/nvidia/OpenMathReasoning)


In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")
dataset = dataset.to_pandas()[
    ["expected_answer", "problem", "generated_solution"]
]

# Try converting to number - if not, replace with NaN
is_number = pd.to_numeric(pd.Series(dataset["expected_answer"]), errors = "coerce").notnull()
# Select only numbers
dataset = dataset.iloc[np.where(is_number)[0]]

dataset


Next, we format each dataset row with the structured reasoning and solution tags:


In [ ]:
def format_dataset(x):
    expected_answer = x["expected_answer"]
    problem = x["problem"]

    # Remove generated <think> and </think>
    thoughts = x["generated_solution"]
    thoughts = thoughts.replace("<think>", "").replace("</think>", "")

    # Strip newlines on left and right
    thoughts = thoughts.strip()
    # Add our custom formatting
    final_prompt = \
        reasoning_start + thoughts + reasoning_end + \
        solution_start + expected_answer + solution_end
    return [
        {"role" : "system",    "content" : system_prompt},
        {"role" : "user",      "content" : problem},
        {"role" : "assistant", "content" : final_prompt},
    ]

dataset["Messages"] = dataset.apply(format_dataset, axis = 1)
tokenizer.apply_chat_template(dataset["Messages"][0], tokenize = False)


The first formatted training example contains the system instruction, a radical-equation problem, and the complete worked solution:

We truncate the pre-fine-tuning dataset at half of max_seq_length because we do not want overly long reasoning traces. This filtering step takes approximately two minutes.


In [ ]:
dataset["N"] = dataset["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))

dataset = dataset.loc[dataset["N"] <= max_seq_length/2].copy()
dataset.shape


We then tokenize the messages and convert them into a Hugging Face-compatible Dataset:


In [ ]:
from datasets import Dataset

dataset["text"] = tokenizer.apply_chat_template(dataset["Messages"].values.tolist(), tokenize = False)
dataset = Dataset.from_pandas(dataset)
dataset


### Pre-Fine-Tuning the Response Format

We now pre-fine-tune the model so it learns to follow the custom GRPO formatting:


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 2, # Set this for 1 full training run.
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use this for WandB etc
    ),
)
trainer.train()


Finally, we check whether the model has learned to follow the custom format:


In [ ]:
text = tokenizer.apply_chat_template(
    dataset[0]["Messages"][:2],
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)


The generated response follows the required reasoning and solution structure. Before moving to the GRPO stage, we remove the temporary dataset and clear unused GPU memory:


In [ ]:
del dataset
torch.cuda.empty_cache()
import gc
gc.collect()


In Part 2, we will define the reward functions, fine-tune the model with GRPO, test the fine-tuned model, and save it locally and on the Hugging Face Hub.


## Part 2 — Train, test, and export the model


Enhancing the reasoning abilities of Large Language Models (LLMs) is important for their application in complex tasks. This technical guide continues a practical walkthrough for fine-tuning the Qwen 3 model specifically for reasoning with Group Relative Policy Optimization (GRPO).

Part 1 covered the foundational steps before the fine-tuning loop: introducing GRPO, setting up the computational environment, loading the Qwen 3 base model and tokenizer, and preparing the target dataset.

[Read Part 1: Prepare Qwen 3, configure LoRA, and preprocess the reasoning dataset](https://todatabeyond.com/blog/qwen-3-mathematical-reasoning-fine-tuning-with-grpo-technique-1)

In this second part, we complete the workflow by defining the reward functions used to train the model, fine-tuning and testing it, and finally saving it locally and on the Hugging Face Hub.

## 1. Define GRPO Reward Functions

We will be using Hugging Face’s Open-R1 Math dataset. You can also utilize OpenAI’s famous GSM8K dataset.

- [Open-R1 DAPO Math 17K Processed dataset](https://huggingface.co/datasets/open-r1/DAPO-Math-17k-Processed)
- [OpenAI GSM8K dataset](https://huggingface.co/datasets/openai/gsm8k)


In [ ]:
from datasets import load_dataset
dataset = load_dataset("open-r1/DAPO-Math-17k-Processed", "en", split = "train")
dataset


Let’s look at the first row:


In [ ]:
dataset[0]["prompt"]


In [ ]:
dataset[0]["solution"]


In the GSM8K dataset, we notice all answers like about have a ####, so we extract it. But for the Open R1 dataset, we can skip the below.


In [ ]:
def extract_hash_answer(text):
    # if "####" not in text: return None
    # return text.split("####")[1].strip()
    return text
extract_hash_answer(dataset[0]["solution"])


Let’s map the dataset as we have done before in the first article, and observe the first row:


In [ ]:
dataset = dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["prompt"]},
    ],
    "answer": extract_hash_answer(x["solution"]),
})
dataset[0]


We create a regex format to match the reasoning sections and answers:


In [ ]:
import re
# Add optional EOS token matching
solution_end_regex = r"</SOLUTION>[\s]{0,}" + \
    "(?:" + re.escape(tokenizer.eos_token) + ")?"
match_format = re.compile(
    rf"{reasoning_end}.*?"\
    rf"{solution_start}(.+?){solution_end_regex}"\
    rf"[\s]{{0,}}$",
    flags = re.MULTILINE | re.DOTALL
)
re.compile(r'<end_working_out>.*?<SOLUTION>(.+?)</SOLUTION>[\s]{0,}(?:<\|endoftext\|>)?[\s]{0,}$',
re.MULTILINE|re.DOTALL|re.UNICODE)
match_format.findall(
    "Let me think!<end_working_out>"\
    f"<SOLUTION>\n2\n</SOLUTION>",
)


In [ ]:
match_format.findall(
 "<start_working_out>Let me think!<end_working_out>"\
 f"<SOLUTION> 2 </SOLUTION>\n\n",
)


We now want to create a reward function to match the format exactly — we reward it with 3 points if it succeeds:


In [ ]:
def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Match if format is seen exactly!
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores


If it fails, we want to reward the model if it at least follows the format partially, by counting each symbol:


In [ ]:
def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Count how many keywords are seen - we penalize if too many!
        # If we see 1, then plus some points!
        # No need to reward <start_working_out> since we always prepend it!
        # score += 0.5 if response.count(reasoning_start) == 1 else -1.0
        score += 0.5 if response.count(reasoning_end)   == 1 else -1.0
        score += 0.5 if response.count(solution_start)  == 1 else -1.0
        score += 0.5 if response.count(solution_end)    == 1 else -1.0
        scores.append(score)
    return scores


Finally, we want to extract the generated answer and reward or penalize it! We also reward it based on how close the answer is to the true one via ratios:


In [ ]:
def check_answer(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]
    extracted_responses = [
        guess.group(1)
        if (guess := match_format.search(r)) is not None else None \
        for r in responses
    ]
    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        score = 0
        if guess is None:
            scores.append(-2.0)
            continue
        # Correct answer gets 5 points!
        if guess == true_answer:
            score += 5.0
        # Match if spaces are seen, but less reward
        elif guess.strip() == true_answer.strip():
            score += 3.5
        else:
            # We also reward it if the answer is close via ratios!
            # Ie if the answer is within some range, reward it!
            try:
                ratio = float(guess) / float(true_answer)
                if   ratio >= 0.9 and ratio <= 1.1: score += 2.0
                elif ratio >= 0.8 and ratio <= 1.2: score += 1.5
                else: score -= 2.5 # Penalize wrong answers
            except:
                score -= 4.5 # Penalize
        scores.append(score)
    return scores


Also, sometimes it might not be 1 number as the answer, but like a sentence, for example, “The solution is $20” -> we extract 20. We also remove possible commas, for example,e as in 123,456


In [ ]:
match_numbers = re.compile(
    solution_start + r".*?[\s]{0,}([-]?[\d\.\,]{1,})",
    flags = re.MULTILINE | re.DOTALL
)
print(match_numbers.findall("<SOLUTION>  0.34  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  123,456  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  -0.234  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>17</SOLUTION>"))


We now prepare our main function, which will print out the generated responses and the true answer, along with another reward function that converts text to float via `float` and sees if it’s the same.


In [ ]:
global PRINTED_TIMES
PRINTED_TIMES = 0
global PRINT_EVERY_STEPS
PRINT_EVERY_STEPS = 5

def check_numbers(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]
    extracted_responses = [
        guess.group(1)
        if (guess := match_numbers.search(r)) is not None else None \
        for r in responses
    ]
    scores = []
    # Print only every few steps
    global PRINTED_TIMES
    global PRINT_EVERY_STEPS
    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print(
            '*'*20 + f"Question:\n{question}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}"
        )
    PRINTED_TIMES += 1
    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:
            scores.append(-2.5)
            continue
        # Convert to numbers
        try:
            true_answer = float(true_answer.strip())
            # Remove commas like in 123,456
            guess       = float(guess.strip().replace(",", ""))
            scores.append(3.5 if guess == true_answer else -1.5)
        except:
            scores.append(0)
            continue
    return scores


We will get the top 90% prompt length so we don’t accidentally truncate them! We’ll remove the top 10% long prompts.


In [ ]:
tokenized = dataset.map(
    lambda x: {"tokens" : tokenizer.apply_chat_template(x["prompt"], add_generation_prompt = True, tokenize = True)},
    batched = True,
)
print(tokenizer.decode(tokenized[0]["tokens"]))
tokenized = tokenized.map(lambda x: {"L" : len(x["tokens"])})

import numpy as np
maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length = ", maximum_length)

# Filter only samples smaller than 90% max length
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized


## 2. Qwen 3 Reasoning Fine-Tuning with GRPO

Now we are ready to set up the GRPO Trainer and all configurations to start fine-tuning the model.


In [ ]:
from trl import GRPOConfig, GRPOTrainer

max_prompt_length = maximum_length + 1 # + 1 just in case!
max_completion_length = max_seq_length - max_prompt_length
from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)
training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.01,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 100,
    save_steps = 100,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",
    # For optional training + evaluation
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
)


Let’s run the trainer, and our goal is to see the reward column increase! You might have to wait 150 to 200 steps for any action. You’ll probably get 0 reward for the first 100 steps. Please be patient!


In [ ]:
# For optional training + evaluation
# new_dataset = dataset.train_test_split(test_size = 0.01)
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_exactly,
        match_format_approximately,
        check_answer,
        check_numbers,
    ],
    args = training_args,
    train_dataset = dataset,
    # For optional training + evaluation
    # train_dataset = new_dataset["train"],
    # eval_dataset = new_dataset["test"],
)
trainer.train()


## 3. Testing the Fine-Tuned Model

Now let’s try the model we just trained! First, let’s try the model without any GRPO trained:


In [ ]:
from vllm import SamplingParams

text = "What is the sqrt of 101?"

sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text
output


And now with the LoRA, we just trained with GRPO — we first save the LoRA!


In [ ]:
model.save_lora("grpo_saved_lora")


Then we will verify that the LoRA is trained!


In [ ]:
from safetensors import safe_open
tensors = {}
with safe_open("grpo_saved_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())


Now we load the LoRA and test:


In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "What is the sqrt of 101?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 2048,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text
output


Our reasoning model is much better — it’s not always correct, since we only trained it for an hour or so — it’ll be better if we extend the sequence length and train for longer!

## 4. Saving the Fine-Tuned Model

### 4.1. Saving to float16 for VLLM

We will save the model to float16 directly. Select merged_16bit for float16 or merged_4bit for int4. We will use push_to_hub_merged to upload to your Hugging Face account! You can get your tokens from your Hugging Face account.

[Open your Hugging Face access-token settings](https://huggingface.co/settings/tokens)


In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")
# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")


### 4.2. GGUF / llama.cpp Conversion

To save the model to GGUF / llama.cpp, we can clone llama.cpp and we default save it to q8_0. Use save_pretrained_gguf for local saving and push_to_hub_gguf for uploading to HF. Some supported quant methods (full list on our Wiki page):

- q8_0 — Fast conversion: High resource use, but generally acceptable.
- q4_k_m — Recommended: Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
- q5_k_m — Recommended: Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.


In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")
# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")
# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )


Now, use the model-unsloth.gguf file or model-unsloth-Q4_K_M.gguf file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan here and Open WebUI here.

- [Jan on GitHub](https://github.com/janhq/jan)
- [Open WebUI on GitHub](https://github.com/open-webui/open-webui)


## Next steps

The notebook leaves all export operations behind explicit `if False` guards. Enable only the format you need, provide your own Hugging Face namespace and token securely, and review the generated model before deployment.
